# Credit Default Risk — Expected Value Decision Model
### Phase 1: cleaning the application table

Lenders don't just need to know who's likely to default — they need to know which loans are worth approving. This project predicts the probability of default on consumer loan applications, then turns that probability into an approve/reject decision based on expected profit rather than a fixed cut-off.

**Data:** Home Credit Default Risk (Kaggle) — 307,511 applications plus six related tables of credit history, loaded into a local PostgreSQL database. About 8% of applicants defaulted.

**The project runs in three steps:**
1. **Phase 1** — the main application table on its own: inspect every column in [01_column_checks.ipynb](01_column_checks.ipynb), clean it in [02_phase1_model.ipynb](02_phase1_model.ipynb) **(this notebook)**, and fit a baseline model.
2. **Phase 2** — build features from the six history tables in SQL, one row per applicant, and measure how much they add.
3. **Decision layer** — convert predicted probabilities into expected profit, choose the approval cut-off, and present it in a Power BI dashboard.

Each cleaning decision below comes with the reason behind it; the evidence for each one is in the column checks notebook. Columns that needed no changes aren't documented.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import numpy as np
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_rows", 210)

load_dotenv(find_dotenv())

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME"),
))

df_raw = pd.read_sql("SELECT * FROM application_train", engine)
print(df_raw.shape)

(307511, 122)


In [2]:
df_raw.head()

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,100002,1,Cash loans,M,N,Y,0,"202,500.0000","406,597.5000","24,700.5000",...,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,100003,0,Cash loans,F,N,N,0,"270,000.0000","1,293,502.5000","35,698.5000",...,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,100004,0,Revolving loans,M,Y,Y,0,"67,500.0000","135,000.0000","6,750.0000",...,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,100006,0,Cash loans,F,N,Y,0,"135,000.0000","312,682.5000","29,686.5000",...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,"121,500.0000","513,000.0000","21,865.5000",...,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


# Manual Encoding and Cleaning
## Numeric column sweep
With 106 numeric columns, inspecting each distribution individually isn't practical. Instead all of them are summarised in one table: distinct values, null share, minimum, median and maximum, plus max_over_median - the maximum divided by the median. Sorting on that last column surfaces the heavily right-tailed distributions, where a single value sits far above the typical one, without needing to read 106 separate outputs.  

The metric only means something for unbounded columns. The housing variables are normalised to 0–1, so their maximum is 1 by construction and dividing it by a near-zero median produces a large ratio that reflects the scale rather than any anomaly. Columns with a median of 0, mostly the binary flags, return NaN and sort to the bottom. Filtering to columns with a maximum above 1 leaves the amounts, counts and day fields, where the ratio is informative.  

On that filtered view amt_income_total is the clear standout, which is what prompted the record-level validation documented below. Sorting the same table by distinct-value count instead identifies the zero-variance and near-constant columns, which carry no information and are dropped by rule rather than individually.  

## Inspection method
All categorical columns were inspected with `value_counts` and all numeric columns with the sweep above, to catch unexpected categories, sentinel values and zero-variance fields. Individual outputs are omitted below; only columns requiring a decision are documented.  

## Column decisions
* **Duplicates** - `sk_id_curr` is unique across all 307,511 records; no duplicate rows.
* **name_contract_type** - Converted to Integer type (Cash loans = 0, Revolving loans = 1).
* **code_gender** - Removed rare gender type(`XNA`) that only appears 4 times, Converted to Integer type (F = 0, M = 1).
* **flag_own_car** - Converted to Integer type (N = 0, Y = 1).
* **flag_own_realty** - Converted to Integer type (N = 0, Y = 1).
* **cnt_children** - capped at 5, Only 42 applicants exceed this, too few for a stable per-level default rate, and values up to 19 distort linear model coefficients.
Capping rather than dropping, since these are valid records and removing them would bias the sample.
* **amt_income_total** - Maximum of 117,000,000 against a median of 147,150. The currency is anonymised in this dataset, so absolute magnitude says nothing; extreme values were assessed on their ratio to the median and on whether the rest of the record corroborated them, rather than by a threshold rule.
`sk_id_curr` = 114967 fails on four counts: the income is 795× the median, the applicant is a labourer with secondary education, the credit-to-income ratio is 0.005 against a population median near 3, and they defaulted on an annuity of 26,194 - implausible on an income of that size. The value is most likely 117,000 recorded at 1000×. Set to NaN rather than dropped, since the record's other 121 fields are valid and it belongs to the minority class.  
The next four largest are consistent with their occupations and credit amounts and were kept. The second-largest, 18,000,090, is worth noting: 122× the median and oddly non-round where its neighbours are clean multiples, which may indicate a keying artefact. But the credit amount, occupation and repayment outcome all corroborate the figure, so a single weak signal wasn't treated as grounds for intervention.
* **amt_goods_price** - Price of the financed item; `amt_credit` is higher because it includes fees and insurance. Null for revolving credit where there is no purchase (278 records).
Default rate by price decile runs 4.9%–13.3% but is not monotonic: it peaks at 13.3% in the 373,500–450,000 band (35,052 records, so a real effect rather than noise) then falls. A logistic regression on the raw column fits a single coefficient and can only express one direction, so the mid-range peak is invisible to it. Tree models find that shape without being told. The relationship is not beyond a linear model, it just has to be engineered in - via binning, a polynomial term, or splines - which is worth testing later rather than assumed.
* **credit_over_goods** (`amt_credit / amt_goods_price`) - Constructed as a proxy for fee loading, on the hypothesis that higher markups reflect risk-based pricing. The hypothesis was not tested: quantile binning fails because ~30% of loans sit at exactly 1.0 - the loan equals the purchase price, so no fees were added - leaving decile edges non-unique. Rather than a smooth distribution, the ratio is a large fee-free group plus a spread of fee-bearing loans above it and a small tail below 1.0. That structure suggests a categorical split (fee-free / standard fees / high markup) rather than quantiles, which is how it should be revisited.
* **days_birth** → **age_years** - Recorded as negative days before the application date, which is the convention for every `days_*` column in this dataset rather than an error. Converted to `-days_birth / 365.2425` to give age in years, and the original dropped: a coefficient per year of age is interpretable, one per day is not, and keeping both would leave two perfectly collinear columns. Range 20.5–69.1, no nulls.
* **days_employed** - Negative days convention, but contains a sentinel: 365,243 (~1000 years) appears in 55,374 records where all genuine values are negative. 99.96% of those records are pensioners, so the sentinel encodes "no current employment" rather than a measurement. Replaced with NaN. A missingness indicator was considered and rejected: 55,362 of the 55,374 flagged records are already identifiable via `name_income_type = 'Pensioner'`, so the flag would duplicate information the model already has.
* **days_registration** - Negative days before the application date, the convention for every `days_*` column here. Sign flipped to positive for readability; a monotonic transform, so it changes nothing for the models. Measures how long ago the applicant last changed their registration. No nulls.
* **days_id_publish** - Same convention, sign flipped to positive. Measures how long ago the applicant's identity document was issued or renewed. No nulls.
* **flag_mobil** - Dropped. 307,510 of 307,511 records are 1, so the column has effectively zero variance and no predictive capacity.
* **flag_cont_mobile** - 99.81% ones (574 zeros). Retained rather than dropped: 574 records is enough to estimate from, though it is unlikely to contribute much.
* **occupation_type** - 31% null, but for two different reasons. 55k are pensioners and unemployed, who have no occupation to report. The other 41k are working people whose field just wasn't filled in. Filled both with an `Unknown` category - mode-imputing would hand 96k people the most common job, which seemed worse. Checked whether the blank itself says anything: working applicants with no occupation default at 8.03% vs 8.79% with one, so not really. The lower rate in the null group is just the pensioners (5.39%), and `name_income_type` already flags those.
* **cnt_fam_members** - Capped at 7; above that it's 41 people across nine levels, too thin to learn from. Correlates 0.879 with `cnt_children`, so the two are largely redundant, but kept both since trees can use whichever splits better. Tried the difference between them as an `adults` column, thinking single-parent vs couple might matter - it does, but barely (8.63% vs 7.87%), and `name_family_status` already says the same thing more directly. Dropped it.
* **region_rating_client / region_rating_client_w_city** - Same 1–3 regional risk rating, with and without city adjustment, correlated 0.9508. Kept the city-adjusted version as the more granular of the two, dropped the other.
* **organization_type** - 58 categories. Its `XNA` isn't the same as the one in `code_gender`: there it was 4 unknown rows, here it's 55,374 - the pensioners with the `days_employed` sentinel, who have no employer to name(Changed `XNA` to `No_employed`). Kept as its own category rather than dropped. This is the fourth column marking the same group, alongside `days_employed`, `flag_emp_phone` and `name_income_type`. Several categories have under 100 records, but the levels were kept as they are: regularisation shrinks the rare coefficients in logistic regression, and trees ignore dummies that don't help, so grouping wasn't worth the information it would throw away.
* **Housing measurements** - 14 building measurements, each recorded three ways (`_avg`, `_mode`, `_medi`). Across all 14, the average correlates 0.990–0.998 with the median and 0.966–0.989 with the mode, so the three versions carry the same information. Kept `_avg`, which has the most distinct values and so the most detail, and dropped the other 28 columns. `totalarea_mode` only exists in one version and stays.
* **obs_30_cnt_social_circle** - One applicant stands far above the rest at 348, where the tail otherwise ends around 30. Looked at the record: it's internally consistent — the same person holds the highest value in all four social-circle columns, and the numbers agree with each other — so it's an unusually large circle rather than an error. Left as it is. Capping was ruled out on purpose: clipping the observation count without the default count would leave that row with more defaults than people observed.  
Also built `social_def_rate_30` = defaults ÷ observations, the share of the applicant's circle that defaulted. Being a rate, it isn't affected by circle size, so the extreme record comes out at an ordinary ~10%.
* **flag_document_2–21** - 20 flags for whether a given document was provided. Applied the same rule as `flag_mobil` and `flag_cont_mobile`: dropped any flag where fewer than ~0.1% of applicants (≈307) differ from the rest, since there's too little to learn from. That removes nine - documents 2, 4, 7, 10, 12, 17, 19, 20 and 21, ranging from 2 to 183 applicants - and the cut falls in a natural gap between 183 and 372. The other 11 were kept.
* **amt_req_credit_bureau_qrt** - One record shows 261 credit enquiries in a single two-month window, while no other enquiry column goes above 27 - the full-year window tops out at 25. Checked the record: the same applicant has almost no enquiries in any other window, so the 261 contradicts the rest of their history. Almost certainly a keying error, possibly a stray digit, but there's no way to tell whether the real value was 21, 26 or something else, so it was set to NaN rather than replaced with a guess. The other five enquiry columns were clean.
* **fondkapremont_mode / housetype_mode / wallsmaterial_mode / emergencystate_mode** - Roughly half to two-thirds missing, for the same reason as the rest of the housing block. Filled with an `Unknown` category rather than the most common value, which would have stamped one building type onto about half the applicants.
* **fondkapremont_mode** - The name is a transliteration of the Russian "фонд капремонта", the building's capital repair fund. The data is anonymised, but a column name like that hints at which market it comes from.

The column-by-column checks behind these decisions are in [01_column_checks.ipynb](01_column_checks.ipynb).

#### Cleaning

In [3]:
df = df_raw.copy()

drop_cols = [
    'days_birth', 'flag_mobil', 'region_rating_client',
    'apartments_mode', 'apartments_medi',
    'basementarea_mode', 'basementarea_medi',
    'years_beginexpluatation_mode', 'years_beginexpluatation_medi',
    'years_build_mode', 'years_build_medi',
    'commonarea_mode', 'commonarea_medi',
    'elevators_mode', 'elevators_medi',
    'entrances_mode', 'entrances_medi',
    'floorsmax_mode', 'floorsmax_medi',
    'floorsmin_mode', 'floorsmin_medi',
    'landarea_mode', 'landarea_medi',
    'livingapartments_mode', 'livingapartments_medi',
    'livingarea_mode', 'livingarea_medi',
    'nonlivingapartments_mode', 'nonlivingapartments_medi',
    'nonlivingarea_mode', 'nonlivingarea_medi',
    'flag_document_2', 'flag_document_4', 'flag_document_7',
    'flag_document_10', 'flag_document_12', 'flag_document_17',
    'flag_document_19', 'flag_document_20', 'flag_document_21',
]

name_contract_type_mapping = {'Cash loans' : 0, 'Revolving loans' : 1}
df['name_contract_type'] = df['name_contract_type'].map(name_contract_type_mapping)

df = df[~df['code_gender'].isin(['XNA'])].reset_index(drop=True)
code_gender_mapping = {'F' : 0, 'M' : 1}
df['code_gender'] = df['code_gender'].map(code_gender_mapping)

flag_own_car_mapping = {'N' : 0, 'Y' : 1}
df['flag_own_car'] = df['flag_own_car'].map(flag_own_car_mapping)

flag_own_realty_mapping = {'N' : 0, 'Y' : 1}
df['flag_own_realty'] = df['flag_own_realty'].map(flag_own_realty_mapping)

df['cnt_children'] = df['cnt_children'].clip(upper=5)

df.loc[df['sk_id_curr'] == 114967, 'amt_income_total'] = np.nan

df['credit_over_goods'] = df['amt_credit'] / df['amt_goods_price']

def fee_band(r):
    if pd.isna(r):  return "no_goods_price"
    if r < 1:       return "deposit_paid"
    if r == 1:      return "no_fees"
    return "fees_added"

df['fee_band'] = df['credit_over_goods'].apply(fee_band)

df['age_years'] = -df['days_birth'] / 365.2425

df.loc[df['days_employed'] == 365243, 'days_employed'] = np.nan
df['days_employed'] = -df['days_employed']

df['days_registration'] = -df['days_registration']

df['days_id_publish'] = -df['days_id_publish']

df['occupation_type'] = df['occupation_type'].fillna('Unknown')

df['cnt_fam_members'] = df['cnt_fam_members'].clip(upper=7)

df['organization_type'] = df['organization_type'].replace('XNA', 'No employer')

df['social_def_rate_30'] = df['def_30_cnt_social_circle'] / df['obs_30_cnt_social_circle']

df["amt_req_credit_bureau_qrt"] = df["amt_req_credit_bureau_qrt"].replace(261, np.nan)

df = df.drop(columns=drop_cols)

for c in ['fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']:
    df[c] = df[c].fillna('Unknown')